In [1]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 3, Finished, Available, Finished, False)

In [2]:
# Remove this cell before running Data Factory Pipeline
start_date = date.today() - timedelta(7)

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 4, Finished, Available, Finished, False)

In [3]:
# Load JSON data i nto a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 5, Finished, Available, Finished, False)

In [4]:
df.head()

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 6, Finished, Available, Finished, False)

Row(geometry=Row(coordinates=[-149.894, 62.072, 45.3], type='Point'), id='aka2026tgnich', properties=Row(alert=None, cdi=None, code='a2026tgnich', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026tgnich&format=geojson', dmin=0.4, felt=None, gap=61, ids=',aka2026tgnich,', mag=1.7, magType='ml', mmi=None, net='ak', nst=35, place='10 km SSW of Susitna North, Alaska', rms=0.6, sig=44, sources=',ak,', status='reviewed', time=1790638898650, title='M 1.7 - 10 km SSW of Susitna North, Alaska', tsunami=0, type='earthquake', types=',origin,phase-data,', tz=None, updated=1790746212512, url='https://earthquake.usgs.gov/earthquakes/eventpage/aka2026tgnich'), type='Feature')

In [5]:
# Reshape earthquake data
df = (
    df
    .select(
        "id",
        col("geometry.coordinates").getItem(0).alias("longitude"),
        col("geometry.coordinates").getItem(1).alias("latitude"),
        col("geometry.coordinates").getItem(2).alias("elevation"),
        col("properties.title").alias("title"),
        col("properties.place").alias("place_description"),
        col("properties.sig").alias("sig"),
        col("properties.mag").alias("mag"),
        col("properties.magType").alias("magType"),
        col("properties.time").alias("time"),
        col("properties.updated").alias("updated")
    )
)

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 7, Finished, Available, Finished, False)

In [6]:
# Validate data: check for misisng or null values
df = (
    df
    .withColumn("longitude", when(isnull(col("longitude")), 0).otherwise(col("longitude")))
    .withColumn("latitude", when(isnull(col("latitude")), 0).otherwise(col("latitude")))
    .withColumn("time", when(isnull(col("time")), 0).otherwise(col("time")))
)

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 8, Finished, Available, Finished, False)

In [7]:
# Convert "time" and "updated" to timestamp
df = (
    df
    .withColumn("time", (col("time") / 1000).cast(TimestampType()))
    .withColumn("updated", (col("updated") / 1000).cast(TimestampType()))
)

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 9, Finished, Available, Finished, False)

In [8]:
# Append to silver table 
df.write.mode("append").saveAsTable("earthquake_events_silver")

StatementMeta(, 198ace0e-9da7-4e14-9742-5a0214ed6f49, 10, Finished, Available, Finished, True)